In [77]:
from functools import reduce

import matplotlib.pyplot as plt
import numpy as np

from core.fullgate import fullgate
from core.fullstate import fullstate
from core.time_evol_solver import create_time_evol_solver as create_solver_v1
from core.time_evol_solver_v2 import create_time_evol_solver as create_solver_v2
from model.physical_system import AbstractResponsivePhysicalSystem, AbstractObservable

In [ ]:
N_QUBIT = 4

INIT_PSI = fullstate(",".join(["z+"] * N_QUBIT))  # 初期状態：|z+⟩
OBSERVABLE = reduce(lambda x, y: x + y, [fullgate(N_QUBIT, f"X{i}") for i in range(N_QUBIT)])


class SimplePhysicalSystem(AbstractResponsivePhysicalSystem):
    """シンプルな物理システム"""

    def __init__(self):
        pass

    def create_hamiltonian(self, u_t=None):
        rng = np.random.default_rng(seed=0)

        y_coeffs = rng.normal(0, 1, N_QUBIT)
        # 2次元の相互作用係数行列を作成（対角成分もそのまま利用）
        xx_coeffs = rng.normal(0, 1, (N_QUBIT, N_QUBIT))
        yy_coeffs = rng.normal(0, 1, (N_QUBIT, N_QUBIT))
        zz_coeffs = rng.normal(0, 1, (N_QUBIT, N_QUBIT))

        y_terms = [y_coeffs[i] * fullgate(N_QUBIT, f"Y{i}") for i in range(N_QUBIT)]
        xx_terms = [
            xx_coeffs[i, j] * fullgate(N_QUBIT, f"X{i},X{j}")
            for i in range(N_QUBIT) for j in range(i + 1, N_QUBIT)
        ]
        yy_terms = [
            yy_coeffs[i, j] * fullgate(N_QUBIT, f"Y{i},Y{j}")
            for i in range(N_QUBIT) for j in range(i + 1, N_QUBIT)
        ]
        zz_terms = [
            zz_coeffs[i, j] * fullgate(N_QUBIT, f"Z{i},Z{j}")
            for i in range(N_QUBIT) for j in range(i + 1, N_QUBIT)
        ]

        return reduce(lambda x, y: x + y, y_terms + xx_terms + yy_terms + zz_terms)


class SimpleObservable(AbstractObservable):
    """シンプルなオブザーバブル"""

    def __init__(self):
        pass

    def create_hamiltonian(self):
        return [OBSERVABLE]


In [79]:
import tracemalloc
from functools import wraps


def measure_memory_usage(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        tracemalloc.start()
        result = func(*args, **kwargs)
        current, peak = tracemalloc.get_traced_memory()
        print(
            f"[{func.__name__}] memory usage: current={current / 1024:.2f} KB, peak={peak / 1024:.2f} KB")
        tracemalloc.stop()
        return result

    return wrapper


@measure_memory_usage
def run_v1_solver(system, observable, t_seq):
    """V1ソルバーを実行して期待値を取得"""
    print("Running V1 solver (100 steps, all results)...")

    solver = create_solver_v1(
        system=system,
        observable=observable,
        collapse_operator=None,
        init_psi=INIT_PSI,
    )

    solver.reset_state()
    result = solver.forward(lambda t: 0, t_seq)
    expect = result.expect(0)

    print(f"V1 completed: {len(expect)} expectation values")
    return expect


@measure_memory_usage
def run_v2_solver(system, t_seq, result_indices):
    """V2ソルバーを実行して選択的に期待値を取得"""
    print("Running V2 solver (100 steps, every 10th result)...")
    print(f"Result indices: {result_indices}")

    solver = create_solver_v2(
        system=system,
        collapse_operator=None,
        init_psi=INIT_PSI,
    )

    solver.reset_state()
    result = solver.forward(lambda t: 0, t_seq, result_index=result_indices)
    expect = result.expect(OBSERVABLE)
    std = result.std(OBSERVABLE)
    times = result.times

    print(f"V2 completed: {len(expect)} expectation values at selected times")
    return expect, std, times


@measure_memory_usage
def get_analytical_solution(system, observable, t_seq):
    """
    解析解：numpyを使って期待値を計算

    |ψ(t)⟩ = exp(-iHt)|ψ(0)⟩
    ⟨O⟩(t) = ⟨ψ(t)|O|ψ(t)⟩
    """
    from scipy.linalg import expm

    print("Calculating analytical solution...")

    # システムのハミルトニアンとオブザーバブルを取得
    hamiltonian_qobj = system.create_hamiltonian()
    obs_qobj = observable.create_hamiltonian()[0]

    # qutip QobjからnumpyArrayに変換
    H = hamiltonian_qobj.full()  # ハミルトニアン行列
    O = obs_qobj.full()  # オブザーバブル行列

    # 初期状態をnumpy配列に変換
    psi_0 = INIT_PSI.full().flatten()  # |z+⟩ = [1, 0]

    # 各時刻での期待値を計算
    expect_values = []
    std_values = []
    for t in t_seq:
        # 時間発展演算子: U(t) = exp(-iHt)
        U_t = expm(-1j * H * t)

        # 時間発展した状態: |ψ(t)⟩ = U(t)|ψ(0)⟩
        psi_t = U_t @ psi_0

        # 期待値: ⟨O⟩(t) = ⟨ψ(t)|O|ψ(t)⟩ = psi_t† O psi_t
        expect_val = np.conj(psi_t) @ O @ psi_t
        expect_values.append(expect_val.real)

        # 標準偏差: ⟨O^2⟩(t) - ⟨O⟩(t)^2
        expect_val_squared = np.conj(psi_t) @ O @ O @ psi_t
        std_val = np.sqrt(expect_val_squared.real - expect_val.real ** 2)
        std_values.append(std_val)

    expect = np.array(expect_values)
    std = np.array(std_values)
    print(f"Analytical solution computed using numpy")
    return expect, std

In [80]:
# === 共通設定：System, Observable, 初期状態 ===
print("Setting up common components...")

# 定数：時刻ステップ数と結果インデックス間隔
NUM_T_SEQ_STEPS = 1000
RESULT_INDEX_STEP = 50

# 時刻配列（0~1, 100ステップ）
t_seq = np.linspace(0, 10.0, NUM_T_SEQ_STEPS)

# システムとオブザーバブルのインスタンス作成
system = SimplePhysicalSystem()
observable = SimpleObservable()

print(f"Time sequence: {len(t_seq)} points from {t_seq[0]} to {t_seq[-1]}")

# === 各ソルバーの実行 ===
# 10ステップごとのインデックス：[0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
result_indices = list(range(0, NUM_T_SEQ_STEPS, RESULT_INDEX_STEP))

# 各処理を関数で実行
expect_v1 = run_v1_solver(system, observable, t_seq)
expect_v2, std_v2, t_v2 = run_v2_solver(system, t_seq, result_indices)
expect_analytical, std_analytical = get_analytical_solution(system, observable, t_seq)

# === 結果の比較 ===
print("\nComparing results...")

# V1とV2の選択的時刻での比較
expect_v1_selected = expect_v1[result_indices]
diff_v1_v2 = np.max(np.abs(expect_v1_selected - expect_v2))
diff_v1_analytical = np.max(np.abs(expect_v1 - expect_analytical))

print(f"Max difference V1 vs V2 (selected times): {diff_v1_v2:.2e}")
print(f"Max difference V1 vs analytical: {diff_v1_analytical:.2e}")

# === 1つのプロットで全て重ねて表示 ===
print("\nCreating visualization...")

plt.figure(figsize=(12, 8))

# V1結果（全時刻、線グラフ）
plt.plot(t_seq, expect_v1, 'b-', linewidth=2, label=f'V1 (all {NUM_T_SEQ_STEPS} steps)', alpha=0.8)

# V2結果（10ステップごと、マーカー）
plt.plot(t_v2, expect_v2, 'ro', markersize=8, label=f'V2 (every {RESULT_INDEX_STEP}th step)',
         alpha=0.8)

# 解析解（破線）
plt.plot(t_seq, expect_analytical, 'k--', linewidth=3, label='Analytical solution', alpha=0.6)

# グラフの設定
plt.xlabel('Time', fontsize=14)
plt.ylabel('⟨σ_y⟩', fontsize=14)
plt.title('Time Evolution Solver Comparison\n' +
          'Initial: |z+⟩, H = σ_x, Observable = σ_y', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# 統計情報をテキストボックスで表示
textstr = f"""Statistics:
V1 points: {len(expect_v1)}
V2 points: {len(expect_v2)}
Max |V1-V2|: {diff_v1_v2:.1e}
Max |V1-Analytical|: {diff_v1_analytical:.1e}"""

props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.text(0.02, 0.98, textstr, transform=plt.gca().transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

# Y軸の範囲を調整（期待値が0近辺の場合のため）
y_max = max(np.max(np.abs(expect_v1)), np.max(np.abs(expect_v2)), 1e-15)
plt.ylim(-y_max * 1.1, y_max * 1.1)

plt.tight_layout()
plt.show()

# === 標準偏差の比較プロット ===
print("\nCreating std comparison plot...")

plt.figure(figsize=(12, 6))

# std_analytical（全時刻、線グラフ）
plt.plot(t_seq, std_analytical, 'g-', linewidth=2,
         label=f'Analytical std (all {NUM_T_SEQ_STEPS} steps)', alpha=0.8)

# std_v2（10ステップごと、マーカー）
plt.plot(t_v2, std_v2, 'ms', markersize=8, label=f'V2 std (every {RESULT_INDEX_STEP}th step)',
         alpha=0.8)

plt.xlabel('Time', fontsize=14)
plt.ylabel('Standard deviation', fontsize=14)
plt.title('Standard Deviation Comparison\n' +
          'Initial: |z+⟩, H = σ_x, Observable = σ_y', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

# Y軸の範囲を調整
std_max = max(np.max(np.abs(std_analytical)), np.max(np.abs(std_v2)), 1e-15)
plt.ylim(-std_max * 1.1, std_max * 1.1)

plt.tight_layout()
plt.show()

# === 詳細結果の表示 ===
print("\nDetailed Results:")
print(f"V1 expectation values - min: {np.min(expect_v1):.6f}, max: {np.max(expect_v1):.6f}")
print(f"V2 expectation values - min: {np.min(expect_v2):.6f}, max: {np.max(expect_v2):.6f}")
print(f"Analytical expectation: {expect_analytical[0]:.6f} (constant)")

print(f"\nFirst 5 V1 values: {expect_v1[:5]}")
print(f"All V2 values: {expect_v2}")
print(f"V2 times: {t_v2}")

print(f"\nComparison completed successfully!")

Setting up common components...
Time sequence: 1000 points from 0.0 to 10.0
Running V1 solver (100 steps, all results)...



KeyboardInterrupt

